In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
B = 5
T = 8
d_k = 4
w = 4 
Q = torch.randint(0,2,(B,T,d_k),dtype = float)
K = torch.randint(0,2,(B,T,d_k),dtype = float)
V = torch.randint(0,2,(B,T,d_k),dtype = float)
mask_curr = torch.triu(torch.ones(w,w), diagonal=1).bool()
mask_prev= torch.triu(torch.ones(w,w), diagonal=1).bool()

In [53]:
Q_chunks = [Q[:,j:j+w,:].float() for j in range(0,T,w)] #Q_chunk[0] : (B,w,d_k)
K_chunks = [K[:,j:j+w,:].float() for j in range(0,T,w)]  #K_chunks.T(-2,-1) : (B,d_k,w) 
# @ = ( B,w,d_k) @ (B,d_k,w) -> (B,w,w) @(B,w,d_k) -> (B,w,d_k)
V_chunks = [V[:,j:j+w,:].float() for j in range(0,T,w)]
chunk_curr = tuple([(F.softmax((Q_chunks[i]@K_chunks[i].transpose(-2,-1)/d_k**0.5).masked_fill(mask_curr, float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(len(Q_chunks))])
res = torch.cat(chunk_curr,dim=1).nan_to_num(0) 
res.shape

RuntimeError: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1

In [ ]:
print(Q)

tensor([[[0., 1., 0., 0.],
         [1., 0., 1., 0.],
         [1., 1., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 1.],
         [1., 1., 1., 1.],
         [0., 0., 0., 1.],
         [1., 0., 1., 0.]],

        [[1., 0., 0., 1.],
         [0., 0., 1., 1.],
         [1., 1., 1., 1.],
         [0., 0., 1., 1.],
         [1., 0., 0., 1.],
         [0., 0., 0., 0.],
         [1., 1., 1., 1.],
         [1., 0., 0., 1.]],

        [[0., 0., 0., 1.],
         [0., 0., 1., 0.],
         [1., 0., 1., 1.],
         [0., 1., 1., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 1.],
         [1., 0., 0., 1.],
         [1., 0., 1., 0.]],

        [[1., 0., 0., 1.],
         [0., 0., 1., 1.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 1., 1.],
         [0., 1., 0., 1.],
         [1., 0., 0., 0.],
         [1., 1., 1., 1.]],

        [[0., 0., 1., 0.],
         [1., 0., 1., 1.],
         [1., 1., 0., 0.],
         [1., 0., 1., 0.],
         [1., 1., 0.

In [ ]:
#get [-x2,x1] from [x1,x2]
Q = torch.reshape(Q,(B,T,d_k//2,2))
x1 = Q[:,:,:,0]
x2  = Q[:,:,:,1]
res_x = torch.stack((-x2,x1),dim=-1).reshape(B,T,d_k)
Q = torch.reshape(Q,(B,T,d_k))

theta = torch.tensor([pow(10000,(-2*t)/d_k) for t in range(d_k//2)])
m = torch.arange(T).unsqueeze(1)
m_theta = theta*m
cos_theta = torch.cos(m_theta).repeat_interleave(2,dim=-1)
sin_theta = torch.sin(m_theta).repeat_interleave(2,dim=-1)

res = res_x*sin_theta + Q*cos_theta
res.shape


torch.Size([5, 8, 4])

In [ ]:
#get [-x2,x1] from [x1,x2]
x = torch.reshape(x,(B,T,d_k//2,2))
x1 = x[:,:,:,0]
x2  = x[:,:,:,1]
res_x = torch.stack((-x2,x1),dim=-1).reshape(B,T,d_k)

theta = torch.tensor([pow(10000,(-2*t)/d_k) for t in range(d_k//2)])
m = torch.arange(T).unsqueeze(1)
m_theta = theta*m
cos_theta = torch.cos(m_theta).repeat_interleave(2,dim=-1)
sin_theta = torch.sin(m_theta).repeat_interleave(2,dim=-1)

res = res_x*sin_theta + x*cos_theta
res.shape


RuntimeError: The size of tensor a (32) must match the size of tensor b (4) at non-singleton dimension 1

In [ ]:
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
k_pos-q_pos


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [-1,  0,  1,  2,  3,  4,  5,  6],
        [-2, -1,  0,  1,  2,  3,  4,  5],
        [-3, -2, -1,  0,  1,  2,  3,  4],
        [-4, -3, -2, -1,  0,  1,  2,  3],
        [-5, -4, -3, -2, -1,  0,  1,  2],
        [-6, -5, -4, -3, -2, -1,  0,  1],
        [-7, -6, -5, -4, -3, -2, -1,  0]])

In [ ]:
n_heads = 8
slopes = torch.tensor([pow(2,-8/n_heads)**i for i in range(1,n_heads+1)],dtype=float)

In [212]:
k = T-1
W = torch.randn((2*k+1,d_k),dtype=float)
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
relative = (q_pos-k_pos).clamp(-k,k)+k #The T*T matrix
#assign an embedding to each i_j : 2k+1 embedding 
relative.shape

torch.Size([8, 8])

In [213]:
embed = torch.nn.Embedding(2*k+1,d_k)
R = embed(relative)
R.shape

torch.Size([8, 8, 4])

In [ ]:
res = torch.einsum("bid,ijd->bij",Q.float(),R.float())

tensor([[[ 7.0113e-02,  1.9775e+00, -4.1777e-01,  3.6332e-01,  6.1219e-01,
          -3.5928e-01,  1.4334e+00,  7.0035e-01],
         [ 8.4055e-01, -4.7064e-01,  1.3241e+00,  5.4920e-01,  2.1788e+00,
           1.5153e+00, -2.2613e+00, -1.7116e+00],
         [ 6.4931e-01,  5.5775e-01, -4.0053e-01,  3.3016e+00,  1.3143e-01,
           2.5421e+00,  2.1275e+00, -2.6206e+00],
         [-7.0020e-01, -1.0686e-01,  2.8678e-01, -1.1435e+00,  3.7252e-01,
           1.5142e+00,  9.9312e-01,  9.2045e-01],
         [ 6.1769e-01, -1.9859e+00, -2.2814e+00,  7.6825e-01, -4.4701e-01,
           1.3757e+00, -7.7759e-01,  1.7274e+00],
         [-9.0873e-02,  1.3979e+00, -1.2995e+00, -1.5561e-01,  7.7222e-01,
          -1.5204e+00,  3.7257e+00,  3.1884e-01],
         [ 1.8396e-02,  3.4095e-01,  1.1593e+00, -9.8054e-01, -8.0492e-01,
           2.1448e-01, -1.1199e+00,  4.2408e-01],
         [-1.3642e+00,  1.5695e+00,  4.6921e-01,  2.3589e-01, -1.7056e+00,
          -1.5833e+00,  8.4055e-01, -4.7064e-01]],